# 🧠 Smart Quiz & Performance Analyzer System
### Menu-Driven | Difficulty Levels | Timer | Negative Marking | Progress Tracking
---
> Run cells **top to bottom** once, then use **Cell 8 (Launch Quiz)** to start.

## Cell 1 — Imports & Global State

In [ ]:
# ─── Cell 1: Imports & Global State ───────────────────────────────────────────
import random
import time

# ── Global storage (persists across cells in the same kernel session) ──
scores_history = []   # list of session dicts
current_session = {}  # populated during a quiz run

print("✅ Imports loaded. Global state initialised.")

## Cell 2 — Question Bank
Each question is a dict with keys: `id`, `question`, `options` (list of 4), `answer` (1-4), `topic`, `difficulty` (`easy`/`medium`/`hard`), `marks`.

In [ ]:
# ─── Cell 2: Question Bank ─────────────────────────────────────────────────────
QUESTION_BANK = [
    # ── Python ──
    {"id": 1,  "question": "What is the output of type([]) in Python?",
     "options": ["<class 'tuple'>", "<class 'list'>", "<class 'array'>", "<class 'dict'>"],
     "answer": 2, "topic": "Python", "difficulty": "easy", "marks": 1},

    {"id": 2,  "question": "Which keyword is used to define a function in Python?",
     "options": ["func", "define", "def", "fun"],
     "answer": 3, "topic": "Python", "difficulty": "easy", "marks": 1},

    {"id": 3,  "question": "What does the 'pass' statement do in Python?",
     "options": ["Exits the loop", "Does nothing (placeholder)", "Skips current iteration", "Returns None"],
     "answer": 2, "topic": "Python", "difficulty": "easy", "marks": 1},

    {"id": 4,  "question": "What is the time complexity of accessing an element in a Python dict?",
     "options": ["O(n)", "O(log n)", "O(1)", "O(n²)"],
     "answer": 3, "topic": "Python", "difficulty": "medium", "marks": 2},

    {"id": 5,  "question": "Which of the following creates a generator in Python?",
     "options": ["[x for x in range(5)]", "(x for x in range(5))", "{x for x in range(5)}", "list(range(5))"],
     "answer": 2, "topic": "Python", "difficulty": "medium", "marks": 2},

    {"id": 6,  "question": "What is the output of: print(0.1 + 0.2 == 0.3) in Python?",
     "options": ["True", "False", "Error", "None"],
     "answer": 2, "topic": "Python", "difficulty": "hard", "marks": 3},

    # ── Data Structures ──
    {"id": 7,  "question": "Which data structure uses LIFO order?",
     "options": ["Queue", "Stack", "Tree", "Graph"],
     "answer": 2, "topic": "Data Structures", "difficulty": "easy", "marks": 1},

    {"id": 8,  "question": "What is the worst-case time complexity of binary search?",
     "options": ["O(n)", "O(n²)", "O(log n)", "O(1)"],
     "answer": 3, "topic": "Data Structures", "difficulty": "medium", "marks": 2},

    {"id": 9,  "question": "In a min-heap, which element is always at the root?",
     "options": ["Maximum", "Median", "Minimum", "Last inserted"],
     "answer": 3, "topic": "Data Structures", "difficulty": "hard", "marks": 3},

    # ── OOP ──
    {"id": 10, "question": "Which OOP concept hides internal implementation details?",
     "options": ["Inheritance", "Polymorphism", "Encapsulation", "Abstraction"],
     "answer": 3, "topic": "OOP", "difficulty": "easy", "marks": 1},

    {"id": 11, "question": "What is method overriding?",
     "options": ["Defining two methods with the same name in one class",
                 "A child class redefining a parent class method",
                 "Calling a method multiple times",
                 "Creating a method inside a method"],
     "answer": 2, "topic": "OOP", "difficulty": "medium", "marks": 2},

    {"id": 12, "question": "Which Python dunder method is called when an object is created?",
     "options": ["__new__", "__create__", "__init__", "__start__"],
     "answer": 3, "topic": "OOP", "difficulty": "medium", "marks": 2},

    # ── Algorithms ──
    {"id": 13, "question": "Which sorting algorithm has best average-case complexity O(n log n)?",
     "options": ["Bubble Sort", "Insertion Sort", "Merge Sort", "Selection Sort"],
     "answer": 3, "topic": "Algorithms", "difficulty": "medium", "marks": 2},

    {"id": 14, "question": "What algorithmic paradigm does dynamic programming use?",
     "options": ["Greedy", "Divide and conquer", "Overlapping subproblems & optimal substructure", "Backtracking"],
     "answer": 3, "topic": "Algorithms", "difficulty": "hard", "marks": 3},

    # ── Databases ──
    {"id": 15, "question": "Which SQL clause is used to filter grouped results?",
     "options": ["WHERE", "FILTER", "HAVING", "GROUP"],
     "answer": 3, "topic": "Databases", "difficulty": "medium", "marks": 2},

    {"id": 16, "question": "What does ACID stand for in database transactions?",
     "options": ["Array, Class, Index, Data",
                 "Atomicity, Consistency, Isolation, Durability",
                 "Access, Control, Integrity, Design",
                 "Aggregate, Cluster, Index, Delete"],
     "answer": 2, "topic": "Databases", "difficulty": "hard", "marks": 3},

    # ── Networking ──
    {"id": 17, "question": "Which protocol is used to send emails?",
     "options": ["HTTP", "FTP", "SMTP", "SSH"],
     "answer": 3, "topic": "Networking", "difficulty": "easy", "marks": 1},

    {"id": 18, "question": "What is the purpose of DNS?",
     "options": ["Encrypt web traffic",
                 "Translate domain names to IP addresses",
                 "Route packets between networks",
                 "Assign MAC addresses"],
     "answer": 2, "topic": "Networking", "difficulty": "easy", "marks": 1},
]

ALL_TOPICS  = sorted(set(q["topic"] for q in QUESTION_BANK))
DIFF_MARKS  = {"easy": 1, "medium": 2, "hard": 3}   # marks per difficulty
TIME_LIMITS = {"easy": 20, "medium": 30, "hard": 45} # seconds per question

print(f"✅ Question bank loaded: {len(QUESTION_BANK)} questions across {len(ALL_TOPICS)} topics.")
print(f"   Topics: {', '.join(ALL_TOPICS)}")

## Cell 3 — Input Validation Helper

In [ ]:
# ─── Cell 3: Input Validation Helper ──────────────────────────────────────────
def get_valid_input(prompt, valid_choices, allow_blank=False):
    """
    Repeatedly prompts until a valid choice is entered.
    valid_choices : list of acceptable string values (compared in lowercase).
    allow_blank   : if True, an empty string is returned as-is.
    Returns the validated string.
    """
    valid_lower = [str(v).lower() for v in valid_choices]
    while True:
        raw = input(prompt).strip()
        if allow_blank and raw == "":
            return raw
        if raw.lower() in valid_lower:
            # return in the original case of valid_choices
            idx = valid_lower.index(raw.lower())
            return str(valid_choices[idx])
        print(f"  ⚠️  Invalid input. Choose from: {', '.join(str(v) for v in valid_choices)}")


def get_int_input(prompt, lo, hi):
    """Prompts for an integer in [lo, hi]. Loops on bad input."""
    while True:
        raw = input(prompt).strip()
        if raw.isdigit() and lo <= int(raw) <= hi:
            return int(raw)
        print(f"  ⚠️  Please enter a whole number between {lo} and {hi}.")


print("✅ Input validation helpers ready.")

## Cell 4 — Display Utilities

In [ ]:
# ─── Cell 4: Display Utilities ─────────────────────────────────────────────────
DIVIDER      = "═" * 60
THIN_DIVIDER = "─" * 60

def header(title):
    print(f"\n{DIVIDER}")
    pad = (60 - len(title)) // 2
    print(" " * pad + title)
    print(DIVIDER)

def subheader(title):
    print(f"\n{THIN_DIVIDER}")
    print(f"  {title}")
    print(THIN_DIVIDER)

def blank():
    print()

def bar(label, value, total, width=30):
    """Prints an ASCII progress bar."""
    filled = int((value / total) * width) if total else 0
    bar_str = "█" * filled + "░" * (width - filled)
    pct = (value / total * 100) if total else 0
    print(f"  {label:<22} [{bar_str}] {value}/{total}  ({pct:.0f}%)")

print("✅ Display utilities ready.")

## Cell 5 — Quiz Engine (Timer + Difficulty + Random Selection)

In [ ]:
# ─── Cell 5: Quiz Engine ───────────────────────────────────────────────────────
def select_questions(difficulty_filter, num_questions):
    """
    Returns a random sample of questions.
    difficulty_filter : 'easy' | 'medium' | 'hard' | 'mixed'
    """
    if difficulty_filter == "mixed":
        pool = QUESTION_BANK[:]
    else:
        pool = [q for q in QUESTION_BANK if q["difficulty"] == difficulty_filter]

    if not pool:
        return []

    num_questions = min(num_questions, len(pool))
    return random.sample(pool, num_questions)


def run_quiz(questions, negative_marking, time_limit_per_q):
    """
    Runs the quiz interactively.
    Returns a list of result dicts, one per question.
    """
    results = []

    for idx, q in enumerate(questions, start=1):
        header(f"Question {idx} of {len(questions)}")
        print(f"  Topic      : {q['topic']}")
        print(f"  Difficulty : {q['difficulty'].capitalize()}   "
              f"| Marks: +{q['marks']}" +
              (f" / -{q['marks']//2 or 1}" if negative_marking else ""))
        print(f"  Time limit : {time_limit_per_q}s")
        blank()
        print(f"  Q: {q['question']}")
        blank()

        for i, opt in enumerate(q["options"], start=1):
            print(f"    {i}. {opt}")
        blank()

        # ── Timer start ──
        print(f"  ⏱  You have {time_limit_per_q} seconds. Enter your answer (1-4):")
        t_start = time.time()
        raw = input("  Your answer: ").strip()
        t_elapsed = time.time() - t_start

        # ── Timeout check ──
        if t_elapsed > time_limit_per_q:
            print(f"  ⌛ Time's up! ({t_elapsed:.1f}s) No marks awarded.")
            status = "timeout"
            marks_earned = 0
        elif raw not in ["1","2","3","4"]:
            print(f"  ⚠️  Invalid input — treated as wrong.")
            status = "invalid"
            marks_earned = -(q["marks"] // 2 or 1) if negative_marking else 0
        elif int(raw) == q["answer"]:
            print(f"  ✅ Correct! +{q['marks']} mark(s). ({t_elapsed:.1f}s)")
            status = "correct"
            marks_earned = q["marks"]
        else:
            penalty = q["marks"] // 2 or 1
            if negative_marking:
                print(f"  ❌ Wrong. Correct answer: {q['answer']}. "
                      f"−{penalty} mark(s). ({t_elapsed:.1f}s)")
                marks_earned = -penalty
            else:
                print(f"  ❌ Wrong. Correct answer: {q['answer']}. ({t_elapsed:.1f}s)")
                marks_earned = 0
            status = "wrong"

        results.append({
            "question_id"  : q["id"],
            "topic"        : q["topic"],
            "difficulty"   : q["difficulty"],
            "max_marks"    : q["marks"],
            "marks_earned" : marks_earned,
            "status"       : status,
            "time_taken"   : round(t_elapsed, 2),
        })

        if idx < len(questions):
            input("  Press Enter for next question...")

    return results


print("✅ Quiz engine ready.")

## Cell 6 — Scoring Engine

In [ ]:
# ─── Cell 6: Scoring Engine ────────────────────────────────────────────────────
def compute_score(results):
    """
    Takes the list of per-question result dicts and returns a session dict
    with all score metrics.
    """
    total_questions = len(results)
    correct   = sum(1 for r in results if r["status"] == "correct")
    wrong     = sum(1 for r in results if r["status"] == "wrong")
    timeouts  = sum(1 for r in results if r["status"] == "timeout")
    invalids  = sum(1 for r in results if r["status"] == "invalid")

    raw_score  = sum(r["marks_earned"] for r in results)
    max_score  = sum(r["max_marks"]    for r in results)
    percentage = (raw_score / max_score * 100) if max_score else 0
    percentage = max(0, percentage)   # clamp to 0 (negative marking can push below)
    accuracy   = (correct / total_questions * 100) if total_questions else 0
    avg_time   = (sum(r["time_taken"] for r in results) / total_questions
                  if total_questions else 0)

    # ── Topic-wise breakdown ──
    topic_stats = {}
    for r in results:
        t = r["topic"]
        if t not in topic_stats:
            topic_stats[t] = {"correct": 0, "total": 0, "marks": 0, "max": 0}
        topic_stats[t]["total"]  += 1
        topic_stats[t]["max"]    += r["max_marks"]
        topic_stats[t]["marks"]  += r["marks_earned"]
        if r["status"] == "correct":
            topic_stats[t]["correct"] += 1

    return {
        "timestamp"       : time.strftime("%Y-%m-%d %H:%M"),
        "total_questions" : total_questions,
        "correct"         : correct,
        "wrong"           : wrong,
        "timeouts"        : timeouts,
        "invalids"        : invalids,
        "raw_score"       : raw_score,
        "max_score"       : max_score,
        "percentage"      : round(percentage, 1),
        "accuracy"        : round(accuracy, 1),
        "avg_time"        : round(avg_time, 1),
        "topic_stats"     : topic_stats,
        "results"         : results,
    }


def display_score_summary(session):
    header("📊  Quiz Results")
    print(f"  Date/Time     : {session['timestamp']}")
    print(f"  Questions     : {session['total_questions']}")
    blank()
    bar("Correct",   session['correct'],  session['total_questions'])
    bar("Wrong",     session['wrong'],    session['total_questions'])
    bar("Timed out", session['timeouts'], session['total_questions'])
    blank()
    print(f"  Score         : {session['raw_score']} / {session['max_score']}")
    print(f"  Percentage    : {session['percentage']}%")
    print(f"  Accuracy      : {session['accuracy']}%")
    print(f"  Avg time/Q    : {session['avg_time']}s")
    blank()

    subheader("Topic Breakdown")
    for topic, stats in session["topic_stats"].items():
        pct = (stats["marks"] / stats["max"] * 100) if stats["max"] else 0
        bar(topic, stats["correct"], stats["total"])

print("✅ Scoring engine ready.")

## Cell 7 — Performance Analyser

In [ ]:
# ─── Cell 7: Performance Analyser ─────────────────────────────────────────────
def classify_level(percentage):
    if percentage >= 80:
        return "Advanced",    "🏆"
    elif percentage >= 55:
        return "Intermediate","⚡"
    elif percentage >= 30:
        return "Beginner",    "📘"
    else:
        return "Needs Work",  "🔄"


def weakest_topics(session, top_n=2):
    """Returns topic names sorted by accuracy (ascending)."""
    ranked = sorted(
        session["topic_stats"].items(),
        key=lambda kv: (kv[1]["correct"] / kv[1]["total"]) if kv[1]["total"] else 0
    )
    return [t for t, _ in ranked[:top_n]]


def detect_trend():
    """Looks at last 3 attempts and returns a trend string."""
    if len(scores_history) < 2:
        return None
    recent = [s["percentage"] for s in scores_history[-3:]]
    if recent[-1] > recent[0]:
        return "improving 📈"
    elif recent[-1] < recent[0]:
        return "declining 📉"
    else:
        return "stable ➡️"


def analyse_performance(session):
    header("🔍  Performance Analysis")

    level, icon = classify_level(session["percentage"])
    trend = detect_trend()
    weak  = weakest_topics(session)

    print(f"  {icon}  Level        : {level}")
    print(f"  📊  Score        : {session['percentage']}%")
    print(f"  🎯  Accuracy     : {session['accuracy']}%")
    print(f"  ⏱   Avg time/Q   : {session['avg_time']}s")
    blank()

    # ── Personalised feedback ──
    subheader("💬  Personalised Feedback")

    # Feedback block 1 — overall level
    if level == "Advanced":
        print("  Outstanding! You're clearly comfortable with the material.")
        print("  Challenge yourself with harder difficulty settings next time.")
    elif level == "Intermediate":
        print("  Good effort — you have a solid foundation.")
        print("  A little more focused revision will push you to Advanced.")
    elif level == "Beginner":
        print("  You're building your base — keep going!")
        print("  Try Easy mode to reinforce fundamentals before moving up.")
    else:
        print("  This was a tough round. Review the basics and retry — you'll improve!")

    # Feedback block 2 — weak topics
    if weak:
        blank()
        print(f"  📌  Focus area(s) : {', '.join(weak)}")
        print(f"  Spend extra revision time on these topic(s).")

    # Feedback block 3 — speed
    blank()
    if session["avg_time"] < 8:
        print("  ⚡ You answered very quickly — double-check before submitting!")
    elif session["avg_time"] > 35:
        print("  ⏳ You took quite long per question. Practice will improve speed.")
    else:
        print("  ⏱  Good pacing — your timing is solid.")

    # Feedback block 4 — timeouts
    if session["timeouts"] > 0:
        blank()
        print(f"  ⌛ You timed out on {session['timeouts']} question(s).")
        print("  Try increasing your reading speed or practicing under time pressure.")

    # Feedback block 5 — multi-attempt trend
    if trend:
        blank()
        print(f"  📈  Recent trend  : {trend}")
        if "improving" in trend:
            print("  Great progress across attempts — keep the momentum!")
        elif "declining" in trend:
            print("  Your scores dipped recently — revisit weak areas and try again.")

    blank()
    print(f"  Attempt #{len(scores_history)} logged.")


print("✅ Performance analyser ready.")

## Cell 8 — History & Leaderboard

In [ ]:
# ─── Cell 8: History & Leaderboard ────────────────────────────────────────────
def view_last_score():
    if not scores_history:
        print("  ℹ️  No quiz attempts yet. Start a quiz first!")
        return
    display_score_summary(scores_history[-1])


def view_all_attempts():
    if not scores_history:
        print("  ℹ️  No attempts on record.")
        return
    header("📜  All Attempts")
    for i, s in enumerate(scores_history, start=1):
        level, icon = classify_level(s["percentage"])
        print(f"  #{i:02d}  {s['timestamp']}  |  "
              f"{s['percentage']:>5}%  |  "
              f"{s['raw_score']:>3}/{s['max_score']}  |  "
              f"{icon} {level}")
    blank()


def view_leaderboard(top_n=5):
    if not scores_history:
        print("  ℹ️  No attempts on record.")
        return
    header(f"🏅  Top {top_n} Attempts (Personal Leaderboard)")
    ranked = sorted(scores_history, key=lambda s: s["percentage"], reverse=True)
    medals = ["🥇", "🥈", "🥉"]
    for rank, s in enumerate(ranked[:top_n], start=1):
        medal = medals[rank-1] if rank <= 3 else f"  #{rank}"
        print(f"  {medal}  {s['timestamp']}  —  "
              f"{s['percentage']}%  ({s['raw_score']}/{s['max_score']} marks)")
    blank()


print("✅ History & leaderboard ready.")

## Cell 9 — Quiz Setup Wizard

In [ ]:
# ─── Cell 9: Quiz Setup Wizard ─────────────────────────────────────────────────
def setup_quiz():
    """
    Interactive setup: difficulty, number of questions, negative marking.
    Returns (questions, negative_marking, time_limit_per_q) or None to cancel.
    """
    header("⚙️   Quiz Setup")

    # ── Difficulty ──
    print("  Difficulty levels:")
    print("    1. Easy    (1 mark / 20 s per question)")
    print("    2. Medium  (2 marks / 30 s per question)")
    print("    3. Hard    (3 marks / 45 s per question)")
    print("    4. Mixed   (random mix)")
    print("    0. Cancel")
    blank()
    d = get_valid_input("  Choose difficulty [0-4]: ", ["0","1","2","3","4"])
    if d == "0":
        return None
    diff_map = {"1": "easy", "2": "medium", "3": "hard", "4": "mixed"}
    difficulty = diff_map[d]

    # ── Number of questions ──
    pool_size = (len(QUESTION_BANK) if difficulty == "mixed"
                 else len([q for q in QUESTION_BANK if q["difficulty"] == difficulty]))
    print(f"\n  Available questions for '{difficulty}': {pool_size}")
    num_q = get_int_input(f"  How many questions (1–{pool_size})? ", 1, pool_size)

    # ── Negative marking ──
    blank()
    nm = get_valid_input("  Enable negative marking? (y/n): ", ["y","n"])
    negative_marking = (nm == "y")

    # ── Time limit (use difficulty-based default or custom) ──
    if difficulty == "mixed":
        # use per-question limit based on each question's difficulty
        time_limit = None   # handled in run_quiz
    else:
        time_limit = TIME_LIMITS[difficulty]

    blank()
    print(f"  ✅  Setup complete!")
    print(f"      Difficulty      : {difficulty.capitalize()}")
    print(f"      Questions       : {num_q}")
    print(f"      Negative marks  : {'Yes' if negative_marking else 'No'}")
    if time_limit:
        print(f"      Time per Q      : {time_limit}s")
    else:
        print(f"      Time per Q      : varies by difficulty")

    questions = select_questions(difficulty, num_q)

    # For mixed mode, attach per-question time limit
    if difficulty == "mixed":
        for q in questions:
            q["_time_limit"] = TIME_LIMITS[q["difficulty"]]
        effective_time = None  # signal to run_quiz to use per-question limit
    else:
        effective_time = time_limit

    return questions, negative_marking, effective_time


print("✅ Quiz setup wizard ready.")

## Cell 10 — Mixed-Mode Quiz Runner (handles per-question timers)

In [ ]:
# ─── Cell 10: Unified Quiz Runner ──────────────────────────────────────────────
def run_quiz_smart(questions, negative_marking, global_time_limit):
    """
    Wraps run_quiz to handle mixed mode (per-question time limits).
    If global_time_limit is None, uses q['_time_limit'] for each question.
    """
    if global_time_limit is not None:
        return run_quiz(questions, negative_marking, global_time_limit)

    # Mixed mode — run one question at a time with individual limits
    all_results = []
    for i, q in enumerate(questions):
        tl = q.get("_time_limit", 30)
        partial = run_quiz([q], negative_marking, tl)
        all_results.extend(partial)
        if i < len(questions) - 1:
            input("  Press Enter for next question...")
    return all_results


print("✅ Unified quiz runner ready.")

## Cell 11 — Main Menu (Entry Point)
▶️ **Run this cell to start the quiz system.**

In [ ]:
# ─── Cell 11: Main Menu ────────────────────────────────────────────────────────
def main_menu():
    global current_session

    header("🧠  SMART QUIZ & PERFORMANCE ANALYZER")
    print("  Welcome! Navigate the menu below.")
    print(f"  Sessions played this run: {len(scores_history)}")

    while True:
        subheader("MAIN MENU")
        print("    1. Start Quiz")
        print("    2. View Last Score")
        print("    3. Performance Analysis (last attempt)")
        print("    4. View All Attempts")
        print("    5. Leaderboard")
        print("    6. Exit")
        blank()

        choice = get_valid_input("  Enter option [1-6]: ", ["1","2","3","4","5","6"])

        # ── 1. Start Quiz ──────────────────────────────────────────────────────
        if choice == "1":
            setup = setup_quiz()
            if setup is None:
                print("  Cancelled — returning to menu.")
                continue

            questions, negative_marking, time_limit = setup

            print("\n  🚀  Starting quiz... Good luck!")
            input("  Press Enter when ready...")

            results = run_quiz_smart(questions, negative_marking, time_limit)
            session = compute_score(results)
            scores_history.append(session)
            current_session = session

            display_score_summary(session)
            analyse_performance(session)

        # ── 2. View Last Score ─────────────────────────────────────────────────
        elif choice == "2":
            view_last_score()

        # ── 3. Performance Analysis ────────────────────────────────────────────
        elif choice == "3":
            if not scores_history:
                print("  ℹ️  Complete at least one quiz first.")
            else:
                analyse_performance(scores_history[-1])

        # ── 4. All Attempts ────────────────────────────────────────────────────
        elif choice == "4":
            view_all_attempts()

        # ── 5. Leaderboard ─────────────────────────────────────────────────────
        elif choice == "5":
            view_leaderboard()

        # ── 6. Exit ────────────────────────────────────────────────────────────
        elif choice == "6":
            header("👋  Goodbye!")
            print(f"  You completed {len(scores_history)} quiz attempt(s) this session.")
            if scores_history:
                best = max(scores_history, key=lambda s: s["percentage"])
                print(f"  Your best score: {best['percentage']}% on {best['timestamp']}")
            blank()
            print("  Keep learning! 🚀")
            print(DIVIDER)
            break


# ── Launch ──
main_menu()